# Data Preparation

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
pd.options.mode.chained_assignment = None



#### Reading all data

In [6]:
aisles = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\aisles.csv')
departments = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\departments.csv')
order_products__prior = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__prior.csv')
order_products__train = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\order_products__train.csv')
orders = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\orders.csv')
products = pd.read_csv(r'C:\Users\MHMD RAGAB\Downloads\DATA\New folder\products.csv')

In [12]:
root = r"C:\Users\MHMD RAGAB\Downloads\DATA\New folder\\"

In [13]:
import pandas as pd
import numpy as np

orders = pd.read_csv(root + 'orders.csv', dtype={
    'order_id': np.int32,
    'user_id': np.int32,
    'eval_set': 'category',
    'order_number': np.int16,
    'order_dow': np.int8,
    'order_hour_of_day': np.int8,
    'days_since_prior_order': np.float32
})

order_products_train = pd.read_csv(root + 'order_products__train.csv', dtype={
    'order_id': np.int32,
    'product_id': np.uint16,
    'add_to_cart_order': np.int16,
    'reordered': np.int8
})

order_products_prior = pd.read_csv(root + 'order_products__prior.csv', dtype={
    'order_id': np.int32,
    'product_id': np.uint16,
    'add_to_cart_order': np.int16,
    'reordered': np.int8
})

products = pd.read_csv(root + 'products.csv', dtype={
    'product_id': np.uint16,
    'aisle_id': np.uint8,
    'department_id': np.uint8
})

aisles = pd.read_csv(root + 'aisles.csv')
departments = pd.read_csv(root + 'departments.csv')

# Features
product_features = pd.read_pickle(root + 'product_features.pkl')
user_features = pd.read_pickle(root + 'user_features.pkl')
user_product_features = pd.read_pickle(root + 'user_product_features.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\MHMD RAGAB\\Downloads\\DATA\\New folder\\\\product_features.pkl'

#### merging train order data with orders

In [ ]:
train_orders = orders.merge(order_products_train, on = 'order_id', how = 'inner')
train_orders.head()

removing unnecessary columns from train_orders

In [ ]:
train_orders.drop(['eval_set', 'add_to_cart_order', 'order_id'], axis = 1, inplace = True)

unique user_ids in train data

In [ ]:
train_users = train_orders.user_id.unique()
train_users[:10]

keeping only train_users in the data

In [ ]:
user_product_features.shape

In [ ]:
user_product_features.head()

In [ ]:
df = user_product_features[user_product_features.user_id.isin(train_users)]
df.head()

In [ ]:
df = df.merge(train_orders, on = ['user_id', 'product_id'], how = 'outer')
df.head()

for order_number, order_dow, order_hour_of_day, days_since_prior_order, impute null values with mean values grouped by users as these products will also be potential candidate for order.

In [ ]:
df.order_number.fillna(df.groupby('user_id')['order_number'].transform('mean'), inplace = True)
df.order_dow.fillna(df.groupby('user_id')['order_dow'].transform('mean'), inplace = True)
df.order_hour_of_day.fillna(df.groupby('user_id')['order_hour_of_day'].transform('mean'), inplace = True)
df.days_since_prior_order.fillna(df.groupby('user_id')['days_since_prior_order'].\
                                                             transform('mean'), inplace = True)

Removing those products which were bought the first time in last order by a user

In [ ]:
df.reordered.value_counts()

In [ ]:
df.reordered.isnull().sum()

In [ ]:
df = df[df.reordered != 0]

In [ ]:
df.shape

Now imputing 0 in reordered as they were not reordered by user in his/her last order.

In [ ]:
df.reordered.fillna(0, inplace = True)

df.isnull().sum()

In [ ]:
df.head()

#### Merging product and user features

In [ ]:
product_features.head()

In [ ]:
user_features.head()

In [ ]:
df = df.merge(product_features, on = 'product_id', how = 'left')
df = df.merge(user_features, on = 'user_id', how = 'left')
df.head()

The dataframe has null values because the product was never bought earlier by a user

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sort_values(ascending = False)

In [ ]:
df.to_pickle(root + 'Finaldata.pkl')

In [ ]:
df2 = pd.read_pickle(root +'Finaldata.pkl')
df2.head()

Yayyyyy. Ready for some cool modeling now :p